# Imports

In [ ]:
from fastai.vision.all import *
from sklearn.model_selection import RepeatedKFold, GroupKFold, GroupShuffleSplit, \
                                    StratifiedGroupKFold, LeaveOneGroupOut, LeavePGroupsOut
from sklearn.utils import resample
import sklearn.metrics as skm
from pathlib import Path
import numpy as np
from numpy import random
import shutil
import glob
import os
from tqdm import tqdm
import pandas as pd
from torch import cuda
import gc
import shutil
import time
state = 36
scratch = os.getenv('SLURM_SCRATCH')
print(scratch)

In [ ]:
torch.cuda.current_device()

# Flush GPU memory if error etc.

In [ ]:
try:
    del learn
except:
    print('no learner')
gc.collect()
torch.cuda.empty_cache()
!nvidia-smi

# Evaluate different model performances with different learning rates etc (copy data to scratch)

In [ ]:
df = pd.read_csv('/path/to/balanced_10000_sk_lu_lv_cr_df_v1_tiles.tsv',
                 sep = '\t')  ## user is responseible to update this tsv file name accordingly
n = 1000 #small for debugging
df.loc[:,'tissue_anno'] = df.tissue + df.anno
df_ds = resample(df,n_samples=n,
             random_state=state, 
             replace=False,
             stratify = df.tissue_anno)
df_ds = df_ds.reset_index(drop=True)
print(df_ds.shape)
df_ds.loc[:,'scratch_fn'] = scratch + '/' + df_ds.fn.str.split('/').str[-1]
for i,fn in enumerate(tqdm(df_ds.fn.values)):
    scratch_fn = df_ds.loc[i,'scratch_fn']
    try:
        shutil.copyfile(fn,scratch_fn)
    except:
        print(scratch_fn,'Copy Failed')
print(df_ds.scratch_fn.isna().sum(),'missing')
df_ds.groupby(['tissue','anno'])['fn'].count()


In [ ]:
state = 36
splitter = TrainTestSplitter(test_size=0.1, random_state=state, stratify=df_ds.tissue_anno.values,
                    train_size=None, shuffle=True)
batch_size = 32 #250 #5000 seemed too high, resnet18->1000-2000 works, smaller for larger models
                #32 for densenet121 seems to be more predictable w/ learning rates..
                #32 works for densenet169, 128 works, 512 does not work
tissue =DataBlock(blocks=(ImageBlock, CategoryBlock),
                  get_x=ColReader('scratch_fn'),
                  splitter=splitter,
                  get_y=  ColReader('anno'),
                  item_tfms=Resize(460), #Presize
                  batch_tfms=aug_transforms(size=299, #299 for inception? 224
                                            max_rotate=45, # size=224,
                                            min_scale=1,
                                            max_zoom=0,
                                            flip_vert=True,
                                           )
                             ) 
dls = tissue.dataloaders(df_ds, bs = batch_size)
dls.show_batch()

In [ ]:
def my_loss(preds,target):
    a,b = preds
    return F.cross_entropy(a,target)

In [ ]:
learn = vision_learner(dls, inception_v3, loss_func = my_loss, metrics= accuracy)
learn.fit_one_cycle(10)
learn.unfreeze()
learn.fit_one_cycle(20,lr_max=3e-3)

In [ ]:
learn.fine_tune(5,base_lr=1e-5)

densenet121, batch size of 32-> fit_one_cycle(5,0.001) -> good. then fit_one_cycle(5,lr_max=5e-4)
densenet121, batch_size = 512 -> fit_one_cycle(5,0.001) -> validation loss just keeps increasing. LR too high?

# Training from scratch?

In [ ]:
df = pd.read_csv('/path/to/balanced_10000_sk_lu_lv_cr_df_v1_tiles.tsv',
                 sep = '\t')  ## user is responseible to update this tsv file name accordingly
n = 4000 #small for debugging
df.loc[:,'tissue_anno'] = df.tissue + df.anno
df_ds = resample(df,n_samples=n,
             random_state=state, 
             replace=False,
             stratify = df.tissue_anno)
df_ds = df_ds.reset_index(drop=True)
print(df_ds.shape)
df_ds.loc[:,'scratch_fn'] = scratch + '/' + df_ds.fn.str.split('/').str[-1]
for i,fn in enumerate(tqdm(df_ds.fn.values)):
    scratch_fn = df_ds.loc[i,'scratch_fn']
    try:
        shutil.copyfile(fn,scratch_fn)
    except:
        print(scratch_fn,'Copy Failed')
print(df_ds.scratch_fn.isna().sum(),'missing')
df_ds.groupby(['tissue','anno'])['fn'].count()

In [ ]:
batch_size = 128
state = 36
splitter = TrainTestSplitter(test_size=0.1, random_state=state, stratify=df_ds.tissue_anno.values,
                    train_size=None, shuffle=True)
tissue =DataBlock(blocks=(ImageBlock, CategoryBlock),
                  get_x=ColReader('fn'),
                  splitter=splitter,
                  get_y=  ColReader('anno'),
                  item_tfms=Resize(460), #Presize
                  batch_tfms=[*aug_transforms(size=224,
                                            max_rotate=45, # size=224,
                                            min_scale=1,
                                            max_zoom=0,
                                            flip_vert=True,
                                           ),
                              Normalize.from_stats(*imagenet_stats)]
                             ) 
dls = tissue.dataloaders(df_ds, bs = batch_size)
model = densenet169()
learn_fs = Learner(dls, model, loss_func=CrossEntropyLossFlat(), metrics=accuracy)
learn_fs.fit_one_cycle(20,lr_max=3e-3)

In [ ]:
print(lr_min)

# Best sensitivity outcome
densenet169,batch size =128, fit_one_cycle(20),  learn.fit_one_cycle(4,lr_max=2e-3)

In [ ]:
#Plot a confusion matrix
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(4,4), dpi=60)
upp, low = interp.confusion_matrix()
tn, fp = upp[0], upp[1]
fn, tp = low[0], low[1]
print(tn, fp, fn, tp)
sensitivity = tp/(tp + fn) #True pos / all positive
print('sensitivity',sensitivity)

specificity = tn/(fp + tn)
print('specificity',specificity) #True neg / all negative

print('false positive', fp/ (fp + tn))
print('true positive', tp/(tp + fn) )

In [ ]:
#Plot a confusion matrix
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(normalize=True,
                             figsize=(4,4), dpi=60)
upp, low = interp.confusion_matrix()
tn, fp = upp[0], upp[1]
fn, tp = low[0], low[1]
print(tn, fp, fn, tp)

all_pos = (tp + fn)
sensitivity = tp/all_pos #True positive rate
print('sensitivity',sensitivity)

specificity = tn/(fp + tn) #True negative rate
print('specificity',specificity) #True neg / all negative

print('false positive', fp/ (fp + tn))
print('true positive', tp/all_pos )

In [ ]:
#Plot a confusion matrix
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(normalize=True,
                             figsize=(4,4), dpi=60)
upp, low = interp.confusion_matrix()
tn, fp = upp[0], upp[1]
fn, tp = low[0], low[1]
print(tn, fp, fn, tp)

all_pos = (tp + fn)
sensitivity = tp/all_pos #True positive rate
print('sensitivity',sensitivity)

specificity = tn/(fp + tn) #True negative rate
print('specificity',specificity) #True neg / all negative

print('false positive', fp/ (fp + tn))
print('true positive', tp/all_pos )